# Ophanim continued pretraining
This trains an existing 4B model on unlabelled domain text using next-token loss and a QLoRA adapter. No customer documents. The bundled synthetic corpus is a small starter corpus, not evidence of production readiness. Attach ophanim-cpt-input.zip as a **private** dataset; select one 16 GB GPU and Internet on. Save & Run All for background execution. A completed run still needs held-out task gates and semantic review before release.

In [ ]:
"""Stage verified private inputs whether Kaggle kept or expanded the ZIP."""
import hashlib
import json
import zipfile
from pathlib import Path


def stage_inputs(input_root, output_root):
    inputs = Path(input_root)
    output = Path(output_root).resolve()
    archives = list(inputs.rglob('ophanim-cpt-input.zip'))
    manifests = list(inputs.rglob('bundle_manifest.json'))
    if len(archives) == 1 and not manifests:
        with zipfile.ZipFile(archives[0]) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)):
                raise RuntimeError('Duplicate archive entries')
            checks = json.loads(archive.read('bundle_manifest.json'))
            if set(checks) != set(names) - {'bundle_manifest.json'}:
                raise RuntimeError('Incomplete bundle')
            payload = {name: archive.read(name) for name in checks}
        source = str(archives[0]); form = 'zip'
    elif not archives and len(manifests) == 1:
        # Kaggle's dataset uploader automatically expands ZIP archives.
        folder = manifests[0].parent.resolve()
        checks = json.loads(manifests[0].read_text(encoding='utf-8'))
        payload = {}
        for name in checks:
            file = (folder / name).resolve()
            if not file.is_relative_to(folder) or file == folder or chr(92) in name or ':' in name:
                raise RuntimeError('Unsafe input path')
            payload[name] = file.read_bytes()
        source = str(folder); form = 'expanded'
    else:
        raise RuntimeError('Attach exactly one private Ophanim CPT input dataset')

    # Validate the entire bundle before writing or executing any input code.
    for name, data in payload.items():
        target = (output / name).resolve()
        if not target.is_relative_to(output) or target == output or chr(92) in name or ':' in name:
            raise RuntimeError('Unsafe input path')
        if hashlib.sha256(data).hexdigest() != checks[name]:
            raise RuntimeError(f'Corrupted input: {name}')
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f'Refusing to replace different staged input: {name}')
    for name, data in payload.items():
        target = output / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(data)
    return {'input': source, 'form': form, 'verified_files': len(payload)}

import os, pathlib
ROOT=pathlib.Path('/kaggle/working/ophanim')
ROOT.mkdir(exist_ok=True)
os.environ.update(OPHANIM_MODELS_DIR='/kaggle/working/models',HF_HOME='/kaggle/working/models/hf-cache',HF_HUB_CACHE='/kaggle/working/models/hf-cache/hub',OLLAMA_MODELS='/kaggle/working/models/ollama')
print(stage_inputs('/kaggle/input', ROOT))
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
    if token: os.environ['HF_TOKEN']=token
except Exception: pass


In [ ]:
# CUDA 12.4 supports T4 and P100; use fp16, never bf16.
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r /kaggle/working/ophanim/ml/requirements-training.txt


In [ ]:
import torch, subprocess, sys, datetime
assert torch.cuda.is_available(), 'Choose GPU in Settings'
print('GPU:',torch.cuda.get_device_name(0))
VERSION='cpt-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
RESUME=None # Restore a previous output into models/adapters/VERSION before setting this. Keep its VERSION.
command=[sys.executable,str(ROOT/'ml/scripts/pretrain.py'),'--corpus',str(ROOT/'corpus'),'--version',VERSION]
if RESUME: command+=['--resume',RESUME]
result=subprocess.run(command)
adapter=pathlib.Path('/kaggle/working/models/adapters')/VERSION
print('Training exit status:',result.returncode)


In [ ]:
# Training loss/perplexity is not task quality. Compare untouched task tests.
metrics=json.loads((adapter/'run_metrics.json').read_text()) if (adapter/'run_metrics.json').exists() else {}
if metrics.get('full_training_completed'):
    subprocess.run([sys.executable,str(ROOT/'ml/scripts/evaluate_gpu.py'),'--dataset',str(ROOT/'dataset'),'--corpus',str(ROOT/'corpus'),'--version',VERSION],check=True)
else:
    print('Incomplete run: package the checkpoints, then resume. No release.')


In [ ]:
# Package even an incomplete run; never include the downloaded base/cache.
files=[p for p in adapter.rglob('*') if p.is_file() and p.name!='checksums.json']
assert files, 'No adapter or checkpoint files were produced'
checks={p.relative_to(adapter).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(adapter/'checksums.json').write_text(json.dumps(checks,indent=2))
with zipfile.ZipFile('/kaggle/working/ophanim-output.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in adapter.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(adapter).as_posix())
comparison=json.loads((adapter/'comparison.json').read_text()) if (adapter/'comparison.json').exists() else {}
print('Output ready. Full training:',metrics.get('full_training_completed',False),'Accepted:',comparison.get('accepted',False))
